# Exploratory Analysis

This notebook explores the cleaned Vietnam history datasets and builds an interactive timeline. The main visualization supports category filtering, hover details, and year-range presets so the full historical span can be inspected without losing the modern periods.

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
import plotly.express as px
from IPython.display import HTML, IFrame, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'processed' / 'events_clean.csv').exists():
            return candidate
        if (candidate / 'data' / 'raw' / 'events.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing data/processed or data/raw')

ROOT = find_project_root()
PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR = ROOT / 'data' / 'raw'

events_path = PROCESSED_DIR / 'events_clean.csv'
periods_path = PROCESSED_DIR / 'periods_clean.csv'

if not events_path.exists():
    events_path = RAW_DIR / 'events.csv'
if not periods_path.exists():
    periods_path = RAW_DIR / 'periods.csv'

events = pd.read_csv(events_path, encoding='utf-8-sig')
periods = pd.read_csv(periods_path, encoding='utf-8-sig')

events['year_start'] = pd.to_numeric(events['year_start'], errors='coerce')
events['year_end'] = pd.to_numeric(events['year_end'], errors='coerce')
events['importance_score'] = pd.to_numeric(events['importance_score'], errors='coerce')
periods['start_year'] = pd.to_numeric(periods['start_year'], errors='coerce')
periods['end_year'] = pd.to_numeric(periods['end_year'], errors='coerce')

events.shape, periods.shape

In [ ]:
events[['event_id', 'event_name', 'year_start', 'period_name', 'broad_era', 'category', 'importance_score']].head()

## Timeline Helper

Use `build_event_timeline()` to filter the event data and render a timeline. Leave a filter as `None` to include all values.

In [ ]:
def year_label(year):
    if pd.isna(year):
        return ''
    year = int(year)
    if year < 0:
        return f'{abs(year)} BCE'
    if year == 0:
        return '0'
    return f'{year} CE'



def safe_filename(text):
    text = str(text or 'plot').lower()
    text = re.sub(r'[^a-z0-9]+', '_', text).strip('_')
    return f'{text or "plot"}.html'

def display_plot(fig, filename=None, height=760, open_browser=False):
    """Save a Plotly figure to HTML and embed it with an iframe.

    This avoids Plotly's notebook MIME renderer, which can fail when nbformat
    is unavailable or when the notebook frontend blocks inline scripts.
    """
    if filename is None:
        filename = safe_filename(fig.layout.title.text if fig.layout.title else 'plot')
    output_path = PROCESSED_DIR / filename
    fig.write_html(output_path, include_plotlyjs=True, full_html=True)
    relative_path = Path(os.path.relpath(output_path, Path.cwd())).as_posix()
    display(HTML(f'<p><a href="{relative_path}" target="_blank">Open interactive chart in a new tab</a></p>'))
    display(IFrame(src=relative_path, width='100%', height=height))
    if open_browser:
        import webbrowser
        webbrowser.open(output_path.resolve().as_uri())
    return output_path

def filter_events(category=None, broad_era=None, period=None, min_importance=1, year_min=None, year_max=None):
    filtered = events.copy()
    if category and category != 'All':
        filtered = filtered[filtered['category'].eq(category)]
    if broad_era and broad_era != 'All':
        filtered = filtered[filtered['broad_era'].eq(broad_era)]
    if period and period != 'All':
        filtered = filtered[filtered['period_name'].eq(period)]
    if min_importance is not None:
        filtered = filtered[filtered['importance_score'].fillna(0) >= min_importance]
    if year_min is not None:
        filtered = filtered[filtered['year_start'] >= year_min]
    if year_max is not None:
        filtered = filtered[filtered['year_start'] <= year_max]
    return filtered.sort_values(['year_start', 'event_id'])

def build_event_timeline(category=None, broad_era=None, period=None, min_importance=1, year_min=None, year_max=None, title='Vietnam Through Time: Event Timeline'):
    filtered = filter_events(category, broad_era, period, min_importance, year_min, year_max)
    plot_data = filtered.copy()
    plot_data['year_display'] = plot_data['year_start'].map(year_label)
    plot_data['importance_size'] = plot_data['importance_score'].fillna(1).clip(lower=1) * 4

    fig = px.scatter(
        plot_data,
        x='year_start',
        y='broad_era',
        color='category',
        size='importance_size',
        size_max=24,
        hover_name='event_name',
        hover_data={
            'event_id': True,
            'year_start': False,
            'year_display': True,
            'period_name': True,
            'category': True,
            'importance_score': True,
            'description_short': True,
            'importance_size': False,
            'broad_era': False,
        },
        labels={'year_start': 'Year', 'broad_era': 'Broad era'},
        title=title,
        height=720,
    )

    fig.update_traces(marker=dict(opacity=0.78, line=dict(width=0.5, color='white')))
    fig.update_layout(
        template='plotly_white',
        legend_title_text='Category',
        hoverlabel=dict(align='left'),
        xaxis=dict(
            rangeslider=dict(visible=True),
            rangeselector=dict(
                buttons=[
                    dict(count=100, label='100y', step='year', stepmode='backward'),
                    dict(count=500, label='500y', step='year', stepmode='backward'),
                    dict(count=1000, label='1000y', step='year', stepmode='backward'),
                    dict(step='all', label='All'),
                ]
            ),
        ),
        margin=dict(l=20, r=20, t=70, b=20),
    )
    return fig

## Main Interactive Timeline

The chart below includes a category dropdown and time-window presets. You can also zoom, pan, use the range slider, and click legend items to hide or show categories.

In [ ]:
timeline_data = events.dropna(subset=['year_start']).copy()
timeline_data['year_display'] = timeline_data['year_start'].map(year_label)
timeline_data['importance_size'] = timeline_data['importance_score'].fillna(1).clip(lower=1) * 4

fig = px.scatter(
    timeline_data,
    x='year_start',
    y='broad_era',
    color='category',
    size='importance_size',
    size_max=24,
    hover_name='event_name',
    hover_data={
        'event_id': True,
        'year_start': False,
        'year_display': True,
        'period_name': True,
        'category': True,
        'importance_score': True,
        'description_short': True,
        'importance_size': False,
        'broad_era': False,
    },
    labels={'year_start': 'Year', 'broad_era': 'Broad era'},
    title='Vietnam Through Time: Interactive Event Timeline',
    height=760,
)

categories = [trace.name for trace in fig.data]
category_buttons = [dict(label='All categories', method='update', args=[{'visible': [True] * len(fig.data)}])]
for category in categories:
    category_buttons.append(
        dict(
            label=category,
            method='update',
            args=[{'visible': [trace.name == category for trace in fig.data]}],
        )
    )

time_buttons = [
    dict(label='Full span', method='relayout', args=[{'xaxis.range': [timeline_data['year_start'].min(), timeline_data['year_start'].max()]}]),
    dict(label='State formation onward', method='relayout', args=[{'xaxis.range': [-3000, timeline_data['year_start'].max()]}]),
    dict(label='Early modern to present', method='relayout', args=[{'xaxis.range': [1400, timeline_data['year_start'].max()]}]),
    dict(label='Colonial to present', method='relayout', args=[{'xaxis.range': [1850, timeline_data['year_start'].max()]}]),
    dict(label='Vietnam War era', method='relayout', args=[{'xaxis.range': [1945, 1976]}]),
]

fig.update_traces(marker=dict(opacity=0.78, line=dict(width=0.5, color='white')))
fig.update_layout(
    template='plotly_white',
    legend_title_text='Category',
    hoverlabel=dict(align='left'),
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=[
                dict(count=100, label='100y', step='year', stepmode='backward'),
                dict(count=500, label='500y', step='year', stepmode='backward'),
                dict(count=1000, label='1000y', step='year', stepmode='backward'),
                dict(step='all', label='All'),
            ]
        ),
    ),
    updatemenus=[
        dict(buttons=category_buttons, direction='down', x=0.0, xanchor='left', y=1.16, yanchor='top', showactive=True),
        dict(buttons=time_buttons, direction='down', x=0.24, xanchor='left', y=1.16, yanchor='top', showactive=True),
    ],
    annotations=[
        dict(text='Category', x=0.0, xref='paper', y=1.205, yref='paper', showarrow=False, align='left'),
        dict(text='Time window', x=0.24, xref='paper', y=1.205, yref='paper', showarrow=False, align='left'),
    ],
    margin=dict(l=20, r=20, t=115, b=20),
)

output_html = display_plot(fig, filename='vietnam_interactive_timeline.html', height=780)

In [ ]:
output_html


## Focused Timeline Examples

Use these examples when the full timeline is too dense. Change the arguments to explore any category, era, period, importance level, or year window.

In [ ]:
display_plot(build_event_timeline(min_importance=4, year_min=-3000, title='High-Importance Events From State Formation Onward'), filename='timeline_high_importance.html')

In [ ]:
display_plot(build_event_timeline(category='Military', year_min=1850, title='Military Events From the Colonial Period Onward'), filename='timeline_military_1850_present.html')

## Supporting Summaries

These quick summaries help explain what the timeline is showing and where the dataset is densest.

In [ ]:
events_by_category = (
    events.groupby('category', dropna=False)
    .size()
    .reset_index(name='event_count')
    .sort_values('event_count', ascending=False)
)

category_fig = px.bar(
    events_by_category,
    x='event_count',
    y='category',
    orientation='h',
    title='Event Count by Category',
    labels={'event_count': 'Events', 'category': 'Category'},
    height=460,
)
category_fig.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'})
display_plot(category_fig, filename='eda_events_by_category.html', height=520)

In [ ]:
events_by_period = (
    events.groupby(['period_id', 'period_name'], dropna=False)
    .agg(
        event_count=('event_id', 'count'),
        start_year=('year_start', 'min'),
        end_year=('year_start', 'max'),
    )
    .reset_index()
    .sort_values('start_year')
)

period_count_fig = px.bar(
    events_by_period,
    x='period_name',
    y='event_count',
    hover_data=['period_id', 'start_year', 'end_year'],
    title='Event Count by Historical Period',
    labels={'period_name': 'Period', 'event_count': 'Events'},
    height=520,
)
period_count_fig.update_layout(template='plotly_white', xaxis_tickangle=-45, margin=dict(l=20, r=20, t=70, b=170))
display_plot(period_count_fig, filename='eda_events_by_period.html', height=620)

In [ ]:
events_by_category.to_csv(PROCESSED_DIR / 'eda_events_by_category.csv', index=False, encoding='utf-8-sig')
events_by_period.to_csv(PROCESSED_DIR / 'eda_events_by_period.csv', index=False, encoding='utf-8-sig')

print('Saved EDA outputs:')
print(PROCESSED_DIR / 'vietnam_interactive_timeline.html')
print(PROCESSED_DIR / 'eda_events_by_category.csv')
print(PROCESSED_DIR / 'eda_events_by_period.csv')